# Structured Output in LangChain

This notebook demonstrates how to extract structured responses from LLMs using `with_structured_output()` using:
1. **Pydantic** (`BaseModel`)
2. **TypedDict (Class syntax)**
3. **TypedDict (Function syntax)**

### Step 1: Imports & Model Initialization

In [1]:
import os
from dotenv import load_dotenv
from typing import TypedDict
from typing_extensions import Annotated
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq

load_dotenv()

# Initialize ChatGroq model
model = ChatGroq(model="llama-3.3-70b-versatile")

### Method 1: Using Pydantic (`BaseModel`)
Returns a typed Pydantic object (accessed with dot notation like `result.title`).

In [2]:
class MovieDetails(BaseModel):
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The director of the movie")
    release_year: int = Field(description="The year the movie was released")
    rating: float = Field(description="The IMDb rating of the movie out of 10")
    genres: list[str] = Field(description="List of genres for the movie")
    summary: str = Field(description="A brief summary of the movie plot")

# Bind structured output schema
model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Inception")

print(f"Title: {response.title}")
print(f"Director: {response.director}")
print(f"Year: {response.release_year}")
print(f"Rating: {response.rating} / 10")
print(f"Genres: {', '.join(response.genres)}")
print(f"Summary: {response.summary}")

Title: Inception
Director: Christopher Nolan
Year: 2010
Rating: 8.5 / 10
Genres: Action, Sci-Fi, Thriller
Summary: A thief who steals corporate secrets through dream-sharing technology is given the inverse task of planting an idea into the mind of a CEO.


### Method 2: Using `TypedDict` (Class Syntax)
Returns a standard Python dictionary.

In [3]:
class MovieDict(TypedDict):
    """Schema for movie information."""
    title: Annotated[str, ..., "The title of the movie"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating out of 10"]

# Bind TypedDict schema to model
typed_dict_model = model.with_structured_output(MovieDict)
dict_response = typed_dict_model.invoke("Provide details about Inception")

print(f"Title: {dict_response['title']}")
print(f"Director: {dict_response['director']}")
print(f"Rating: {dict_response['rating']} / 10")
print(f"Dictionary Output: {dict_response}")

Title: Inception
Director: Christopher Nolan
Rating: 8.5 / 10
Dictionary Output: {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception'}


### Method 3: Using `TypedDict` (Function Syntax)
Functional definition using `TypedDict('Name', {...})`.

In [4]:
# Functional TypedDict syntax
PersonInfo = TypedDict('PersonInfo', {
    'name': str,
    'age': int,
    'occupation': str
})

# Bind functional TypedDict schema to model
person_extractor = model.with_structured_output(PersonInfo)
person_res = person_extractor.invoke("John Doe is a 30 year old Software Engineer")

print(f"Name: {person_res['name']}")
print(f"Age: {person_res['age']}")
print(f"Occupation: {person_res['occupation']}")
print(f"Full Dictionary: {person_res}")

Name: John Doe
Age: 30
Occupation: Software Engineer
Full Dictionary: {'age': 30, 'name': 'John Doe', 'occupation': 'Software Engineer'}
